# InvoiceAI — Quickstart

Extract structured data from receipts and invoices with **Qwen2.5-VL-3B** + **PaddleOCR models (via RapidOCR)**.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

**After a restart or disconnect:** run the cells again from the top (`Runtime → Run all`).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"  # <- change this
REPO_DIR = "/content/invoice-ai"

import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/invoice-ai.git {REPO_DIR}
!ls {REPO_DIR}

## 3. Install requirements

Takes 2-4 minutes. You can ignore warnings about `gradio` or `diffusers`.

If Colab asks you to **restart the session**, click Restart and then continue with step 4.

In [ ]:
!pip install -q -r /content/invoice-ai/requirements.txt

## 4. Load the models

First run downloads the VLM (~7 GB). The OCR models come with the pip package. Takes a few minutes.

If the model output is empty or looks like `!!!!!!`, add `os.environ["INVOICEAI_DTYPE"] = "bfloat16"` at the top of this cell, restart the session and run again.

In [ ]:
import os, sys, json, time, logging
os.chdir("/content/invoice-ai")
sys.path.insert(0, "/content/invoice-ai")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", stream=sys.stdout, force=True)

from app.config import settings
from app.pipeline import get_pipeline

print(settings)
pipeline = get_pipeline()

start = time.time()
pipeline.load()
print(f"Models loaded in {time.time() - start:.1f} s")

## 5. Test OCR alone

Checks that OCR works before we run everything.

In [ ]:
from pathlib import Path

sample_files = sorted(p for p in Path("samples").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".pdf"})
print(f"Found {len(sample_files)} sample files")

page = pipeline.load_pages(sample_files[0])[0]
ocr_lines = pipeline.ocr.read(page)
print(f"{len(ocr_lines)} OCR lines. First 10:")
for line in ocr_lines[:10]:
    print(f"  {line.confidence:.2f}  {line.text}")

## 6. Full pipeline on the samples

For each file: the image with boxes, a table of fields with confidence, and the line items.

Colors: 🟩 high (found in OCR text) · 🟨 medium (weak match) · 🟥 low (not found, check it).

In [ ]:
import pandas as pd
from IPython.display import display

from app.extractor import ExtractionError

def show_document(doc, max_width=900):
    """Show the drawn page(s) and the field tables for one processed document."""
    r = doc.result
    print(f"{r.file_name}  |  pages: {r.num_pages}  |  OCR lines: {r.ocr_line_count}  |  time: {r.processing_seconds} s")
    for warning in r.warnings:
        print("  WARNING:", warning)

    for page_index in range(r.num_pages):
        drawn = doc.draw(page_index)
        drawn.thumbnail((max_width, max_width * 3))
        display(drawn)

    rows = [{"field": name, "value": f.value, "confidence": f.confidence, "score": f.match_score}
            for name, f in r.fields.items()]
    display(pd.DataFrame(rows))

    if r.line_items:
        item_rows = [{name: f"{f.value} ({f.confidence})" for name, f in item.items()} for item in r.line_items]
        display(pd.DataFrame(item_rows))

results = []
for path in sample_files[:3]:
    print("=" * 100)
    try:
        doc = pipeline.process_file(path)
        results.append(doc)
        show_document(doc)
    except ExtractionError as exc:
        print(f"FAILED {path.name}: {exc}")
        print("Raw model output:", pipeline.extractor.last_raw_output)

## 7. Test a PDF

We turn one sample image into a PDF, so you can test PDF support without finding a PDF.
You can also upload your own PDF into `samples/` and change the path.

In [ ]:
from PIL import Image

pdf_path = "/content/test_invoice.pdf"
Image.open(sample_files[0]).convert("RGB").save(pdf_path)

doc = pipeline.process_file(pdf_path)
show_document(doc)

## 8. Full JSON result (one document)

In [ ]:
print(doc.result.model_dump_json(indent=2))